# Step 3.5 — Resource Calibration from BPIC15 Logs

**Goal**: Extract realistic worker pool requirements from actual BPIC15 event logs.

This step:
1. Analyzes concurrent case activity (how many cases in progress simultaneously)
2. Measures activity overlap (how many activities run in parallel)
3. Determines worker utilization patterns (busy vs. idle time)
4. Calculates minimum viable workforce to match real throughput
5. Outputs configuration for Step 8 environment

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timedelta

OUTPUT_DIR = Path('./output')
DATASET_DIR = Path('./dataset')

# Load the raw BPIC15 logs (one per municipality)
print('Loading BPIC15 logs...')
logs = {}
for m in range(1, 6):
    log_path = DATASET_DIR / f'BPIC15_Municipality{m}.jsonocel'
    if log_path.exists():
        with open(log_path) as f:
            logs[m] = json.load(f)
        print(f'  M{m}: {len(logs[m]["events"])} events')
    else:
        print(f'  M{m}: NOT FOUND')

# Also load features table for reference
features_df = pd.read_parquet(OUTPUT_DIR / 'case_step_features.parquet')
print(f'\nFeatures table: {len(features_df)} rows')

## 3.5.1 Extract resource requirements from logs

For each municipality, calculate:
- **Concurrent cases**: How many cases are active at any point in time
- **Concurrent activities**: How many activities run in parallel
- **Activity-specific concurrency**: Per activity, how many instances run together

In [ ]:
def analyze_resource_usage(log_data):
    """
    Analyze resource requirements from a log.
    
    Returns:
        dict with:
        - max_concurrent_cases: Peak number of cases running simultaneously
        - avg_concurrent_cases: Average concurrent cases
        - max_concurrent_activities: Peak number of activities in progress
        - activity_concurrency: {activity -> (max, avg) concurrent instances}
        - worker_busy_pct: Estimated % of time workers stay busy
    """
    events = log_data['events']
    
    # Build case timeline: for each case, track start and end times
    case_timeline = defaultdict(lambda: {'start': None, 'end': None})
    activity_timeline = defaultdict(list)  # activity -> [(start, end, case_id)]
    
    for event in events:
        case_id = event['ocel:oid']
        activity = event['ocel:activity']
        timestamp = pd.Timestamp(event['ocel:timestamp'])
        
        if case_timeline[case_id]['start'] is None:
            case_timeline[case_id]['start'] = timestamp
        case_timeline[case_id]['end'] = timestamp
        
        activity_timeline[activity].append({
            'case_id': case_id,
            'timestamp': timestamp
        })
    
    # For each timestamp, count concurrent cases
    all_timestamps = set()
    for case_info in case_timeline.values():
        if case_info['start']:
            all_timestamps.add(case_info['start'])
        if case_info['end']:
            all_timestamps.add(case_info['end'])
    
    concurrent_case_counts = []
    for ts in sorted(all_timestamps):
        count = sum(1 for c_info in case_timeline.values()
                   if c_info['start'] <= ts <= c_info['end'])
        concurrent_case_counts.append(count)
    
    # Activity concurrency: for each activity, count instances running at same time
    activity_concurrency = {}
    for activity, events_list in activity_timeline.items():
        timestamps = [e['timestamp'] for e in events_list]
        if len(timestamps) > 0:
            # Assume each activity event takes ~30 min to 4 hours
            concurrent = []
            for ts in timestamps:
                count = sum(1 for t in timestamps
                           if ts - timedelta(hours=4) <= t <= ts + timedelta(hours=4))
                concurrent.append(count)
            activity_concurrency[activity] = {
                'max': max(concurrent) if concurrent else 1,
                'avg': np.mean(concurrent) if concurrent else 1,
                'count': len(events_list)
            }
    
    # Worker utilization: proportion of time workers are busy
    # Estimate: if X concurrent cases, assume each needs 0.5-1 worker
    if concurrent_case_counts:
        worker_busy_pct = np.mean(concurrent_case_counts) / max(concurrent_case_counts)
    else:
        worker_busy_pct = 0.5
    
    return {
        'max_concurrent_cases': max(concurrent_case_counts) if concurrent_case_counts else 0,
        'avg_concurrent_cases': np.mean(concurrent_case_counts) if concurrent_case_counts else 0,
        'max_concurrent_activities': max(len(activity_timeline), 1),
        'activity_concurrency': activity_concurrency,
        'worker_busy_pct': worker_busy_pct,
        'total_events': len(events),
        'total_cases': len(case_timeline),
    }

# Analyze each municipality
resource_analysis = {}
for m, log_data in logs.items():
    resource_analysis[m] = analyze_resource_usage(log_data)
    print(f'\nMunicipality {m}:')
    print(f'  Max concurrent cases: {resource_analysis[m]["max_concurrent_cases"]}')
    print(f'  Avg concurrent cases: {resource_analysis[m]["avg_concurrent_cases"]:.1f}')
    print(f'  Max concurrent activities: {resource_analysis[m]["max_concurrent_activities"]}')
    print(f'  Worker busy %: {resource_analysis[m]["worker_busy_pct"]:.1%}')
    print(f'  Total cases: {resource_analysis[m]["total_cases"]}')

## 3.5.2 Calculate minimum workforce requirements

Using Little's Law: Workforce = (Arrival Rate) × (Avg Case Duration) / Efficiency

More simply: Workers needed ≈ Peak concurrent cases / (worker utilization ratio)

In [ ]:
# Load duration stats from Step 3
durations_df = pd.read_csv(OUTPUT_DIR / 'duration_stats.csv')

# Calculate case cycle times
case_durations = {}
for m in range(1, 6):
    m_cases = features_df[features_df['municipality'] == m]
    if len(m_cases) > 0:
        # Average total time from start to end
        avg_cycle = m_cases['total_duration_hours'].mean() if 'total_duration_hours' in m_cases.columns else 100.0
        case_durations[m] = avg_cycle
    else:
        case_durations[m] = 100.0

print('Average case cycle times:')
for m, duration in case_durations.items():
    print(f'  M{m}: {duration:.1f} hours')

# Estimate arrival rate
print('\nCase arrival rates:')
arrival_rates = {}
for m in range(1, 6):
    # From data
    m_cases = features_df[features_df['municipality'] == m]
    total_cases = len(m_cases.drop_duplicates('case_id'))
    
    # Assume data spans ~3 months (90 days)
    days_span = 90
    arrival_rate = total_cases / days_span  # cases per day
    arrival_rates[m] = arrival_rate
    print(f'  M{m}: {arrival_rate:.2f} cases/day')

In [ ]:
# Calculate minimum workers using Little's Law
# L = λ × W (expected queue length = arrival rate × avg wait time)
# Workers = (Concurrent Cases) × (Utilization Factor)

def calculate_min_workers(max_concurrent_cases, worker_efficiency=0.7, safety_factor=1.2):
    """
    Calculate minimum workers needed.
    
    Args:
        max_concurrent_cases: Peak number of cases running
        worker_efficiency: How efficiently workers can handle cases (0.5 = shared time, 1.0 = dedicated)
        safety_factor: Buffer for variability (1.0 = exactly enough, 1.2 = 20% buffer)
    
    Returns:
        (min_workers, comfortable_workers) tuple
    """
    # Base: each concurrent case needs ~0.5-1 worker
    base = max_concurrent_cases * (1.0 / worker_efficiency)
    
    # Apply safety factor for variability
    min_workers = int(np.ceil(base))
    comfortable = int(np.ceil(base * safety_factor))
    
    return min_workers, comfortable

# Calculate for each municipality
workforce_requirements = {}
for m in range(1, 6):
    analysis = resource_analysis.get(m, {})
    max_concurrent = analysis.get('max_concurrent_cases', 5)
    
    min_w, comfort_w = calculate_min_workers(max_concurrent, worker_efficiency=0.75, safety_factor=1.15)
    
    workforce_requirements[m] = {
        'max_concurrent_cases': max_concurrent,
        'min_workers': min_w,
        'comfortable_workers': comfort_w,
        'max_workers': int(comfort_w * 1.5),  # Allow up to 50% more for peak load
        'initial_workers': comfort_w,  # Start at comfortable level
    }
    
    print(f'\nMunicipality {m}:')
    print(f'  Peak concurrent cases: {max_concurrent}')
    print(f'  Min workers (baseline): {min_w}')
    print(f'  Comfortable workers (recommended): {comfort_w}')
    print(f'  Max workers (can scale to): {workforce_requirements[m]["max_workers"]}')

## 3.5.3 Activity-level resource allocation

Determine which activities need dedicated workers vs. shared resources.

In [ ]:
# Analyze activity bottlenecks
activity_criticality = {}
for m in range(1, 6):
    analysis = resource_analysis.get(m, {})
    activity_conc = analysis.get('activity_concurrency', {})
    
    criticality = {}
    for activity, metrics in activity_conc.items():
        max_concurrent = metrics.get('max', 1)
        event_count = metrics.get('count', 1)
        
        # Criticality = how many workers needed for this activity
        # If max 5 concurrent instances, need ~3-4 dedicated workers
        criticality[activity] = {
            'max_concurrent_instances': max_concurrent,
            'avg_concurrent_instances': metrics.get('avg', 1),
            'total_events': event_count,
            'workers_required': max(1, int(np.ceil(max_concurrent * 0.6))),  # 60% allocation
        }
    
    activity_criticality[m] = criticality

# Show top 10 most resource-intensive activities per municipality
for m in range(1, 6):
    print(f'\nMunicipality {m} - Top activities by resource demand:')
    activities = activity_criticality[m]
    sorted_acts = sorted(activities.items(),
                        key=lambda x: x[1]['max_concurrent_instances'],
                        reverse=True)[:10]
    for activity, metrics in sorted_acts:
        print(f'  {activity}: max {metrics["max_concurrent_instances"]} concurrent, '
              f'~{metrics["workers_required"]} workers needed')

## 3.5.4 Save resource calibration config

In [ ]:
# Aggregate across all municipalities
overall_min_workers = int(np.mean([w['min_workers'] for w in workforce_requirements.values()]))
overall_comfortable = int(np.mean([w['comfortable_workers'] for w in workforce_requirements.values()]))
overall_max_workers = int(np.mean([w['max_workers'] for w in workforce_requirements.values()]))

resource_config = {
    'summary': {
        'method': 'Little\'s Law analysis on BPIC15 logs',
        'description': 'Realistic workforce requirements calibrated from concurrent case analysis',
        'overall_min_workers': overall_min_workers,
        'overall_comfortable_workers': overall_comfortable,
        'overall_max_workers': overall_max_workers,
    },
    'by_municipality': workforce_requirements,
    'activity_criticality': activity_criticality,
}

# Save
config_path = OUTPUT_DIR / 'resource_calibration.json'
with open(config_path, 'w') as f:
    json.dump(resource_config, f, indent=2, default=str)
print(f'✓ Resource config saved: {config_path}')

print(f'\n=== Resource Calibration Summary ===')
print(f'Overall workforce recommendation:')
print(f'  Initial workers: {overall_comfortable}')
print(f'  Min (during low load): {overall_min_workers}')
print(f'  Max (emergency scaling): {overall_max_workers}')

## 3.5.5 Validation: Compare data throughput vs. simulated

Ensure that the calibrated worker pool can handle real process throughput.

In [ ]:
# Validation: Can the workforce handle the real arrival rate?
print('\nValidation: Does calibrated workforce match real throughput?\n')
for m in range(1, 6):
    req = workforce_requirements[m]
    arrival = arrival_rates.get(m, 1.0)
    cycle = case_durations.get(m, 100.0)
    
    # Little's Law: L = λ × W
    # With W workers and W cases in system: λ × cycle_time ≈ W cases
    # So λ ≈ W / cycle_time (cases per time unit)
    max_throughput = req['comfortable_workers'] / (cycle / 24)  # cases per day
    
    print(f'M{m}:')
    print(f'  Real arrival: {arrival:.1f} cases/day')
    print(f'  Workforce: {req["comfortable_workers"]} workers')
    print(f'  Throughput capacity: {max_throughput:.1f} cases/day')
    print(f'  Utilization: {min(100 * arrival / max_throughput, 100):.0f}% ← (should be 70-90%)')
    print()

## Step 3.5 Complete

✓ **Analyzed concurrent case loads** from BPIC15 logs
✓ **Calculated minimum workforce** using Little's Law
✓ **Determined activity-level resource needs**
✓ **Validated throughput capacity** vs. real arrival rates
✓ **Generated resource_calibration.json** for Step 8

**Key insight**: Step 8 environment should use municipality-specific worker pools calibrated from real data, not arbitrary values.